# Gap & Stone-Separation Metrics (Step 5)

Quantifies what pixel IoU cannot: whether models **separate individual stones**
or merge them, and how well the **inter-stone gaps** are detected. Port of the
archived `04_segmentation_evaluation` notebooks; see `amg_pipeline/gapmetrics.py`.

Per (experiment, variant, run, wall) + pooled AllWalls row:
- **gap metrics**: gap IoU/precision/recall/F1 + predicted/GT gap-pixel ratio
  (>1.2 over-segments, <0.8 under-segments), gaps via closing (disk r=45);
- **stone metrics**: per-GT-stone coverage & separation; detected/merged/
  undetected counts at threshold pairs 0.9/0.9 (primary, as in the archived
  notebook), 0.7/0.7, 0.5/0.5; per-class breakdown at 0.9/0.9.

**Run on the Windows machine** (reads GT masks + RAW rasters; eval-only, CPU).
Runtime is dominated by one morphological closing per raster: the default set
(2 ensembles + 5-run frozen baseline = 84 rasters) ≈ 30–90 min. Progress is
checkpointed per experiment to a .partial.csv; a completed run writes
`experiments/v9_gapstone/gap_stone_metrics.csv` (pull this back).

## 0. Imports

In [1]:
import os, sys, dataclasses
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import amg_pipeline as amg
from amg_pipeline.config import RunConfig
from amg_pipeline.gapmetrics import run_gap_eval
from amg_pipeline import paths

print("amg_pipeline loaded from:", os.path.dirname(amg.__file__))

amg_pipeline loaded from: c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\amg_pipeline


## 1. CONFIG

In [2]:
EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments")
OUT_NAME = "v9_gapstone"

# Which rasters to evaluate: (experiment_name, n_runs)
INCLUDE_COSINE_ENS = True      # v8_coslr_ens for completeness (12 rasters)
INCLUDE_FROZEN_BASELINE = True # per-run spread context (60 rasters, the slow part)
EXPERIMENTS = [("v2_baseline_ens", 1)]
if INCLUDE_COSINE_ENS:
    EXPERIMENTS.append(("v8_coslr_ens", 1))
if INCLUDE_FROZEN_BASELINE:
    EXPERIMENTS.append(("v2_yaw_correction-epsV1", 5))

CHANNEL_VARIANTS = (3, 4, 7)
KERNEL_RADIUS = 45            # same closing radius as the ROI evaluation
MIN_STONE_SIZE = 100          # px, as in the archived notebook

TEST_MASK_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\03_test-masks"
WALLS = ["wall1", "wall2", "wall3", "wall4"]

BASE_CONFIG = RunConfig(
    channels=7, run_number=1,
    experiment_name=EXPERIMENTS[0][0], experiments_root=EXPERIMENTS_ROOT,
    test_mask_dir=TEST_MASK_DIR, walls=WALLS,
    kernel_radius=KERNEL_RADIUS,
)
CSV_PATH = os.path.join(EXPERIMENTS_ROOT, OUT_NAME, "gap_stone_metrics.csv")
print("experiments:", EXPERIMENTS)
print("will write ->", CSV_PATH)

experiments: [('v2_baseline_ens', 1), ('v8_coslr_ens', 1), ('v2_yaw_correction-epsV1', 5)]
will write -> c:\Users\admin\Documents\GitHub\AppearanceMeetsGeometry\experiments\v9_gapstone\gap_stone_metrics.csv


## 2. Run (eval-only; skip-if-exists on the final CSV)

In [3]:
DO_RUN = True    # <-- armed
FORCE = False

if DO_RUN:
    gs = run_gap_eval(BASE_CONFIG, EXPERIMENTS, channel_variants=CHANNEL_VARIANTS,
                      out_name=OUT_NAME, kernel_radius=KERNEL_RADIUS,
                      min_stone_size=MIN_STONE_SIZE, force=FORCE)
else:
    print("DO_RUN is False")

[GT] wall1: 179 stones (>= 100 px), 668,851 gap pixels (kernel r=45)
[GT] wall2: 105 stones (>= 100 px), 3,496,235 gap pixels (kernel r=45)
[GT] wall3: 202 stones (>= 100 px), 369,483 gap pixels (kernel r=45)
[GT] wall4: 177 stones (>= 100 px), 1,193,083 gap pixels (kernel r=45)
[ok] v2_baseline_ens 3ch_run1 wall1: gap IoU 0.437 | det@90/90 0.307
[ok] v2_baseline_ens 3ch_run1 wall2: gap IoU 0.428 | det@90/90 0.162
[ok] v2_baseline_ens 3ch_run1 wall3: gap IoU 0.513 | det@90/90 0.510
[ok] v2_baseline_ens 3ch_run1 wall4: gap IoU 0.403 | det@90/90 0.441
[ok] v2_baseline_ens 4ch_run1 wall1: gap IoU 0.360 | det@90/90 0.067
[ok] v2_baseline_ens 4ch_run1 wall2: gap IoU 0.499 | det@90/90 0.210
[ok] v2_baseline_ens 4ch_run1 wall3: gap IoU 0.486 | det@90/90 0.238
[ok] v2_baseline_ens 4ch_run1 wall4: gap IoU 0.458 | det@90/90 0.605
[ok] v2_baseline_ens 7ch_run1 wall1: gap IoU 0.466 | det@90/90 0.246
[ok] v2_baseline_ens 7ch_run1 wall2: gap IoU 0.533 | det@90/90 0.305
[ok] v2_baseline_ens 7ch_run1 

## 3. Summary — the pixel-vs-stone question

AllWalls (pooled stones / summed gap counts) per experiment and variant.
The hypothesis under test: appearance-only wins pixel metrics but merges
stones; geometry-only separates cleanly; the full model sits between.

In [4]:
if not os.path.exists(CSV_PATH):
    print("No CSV yet."); gs = None
else:
    gs = pd.read_csv(CSV_PATH)
    aw = gs[gs.wall == "AllWalls"]
    cols = ["gap_iou", "gap_f1", "gap_ratio", "detection_rate_c90_s90",
            "merge_rate_c90_s90", "detection_rate_c50_s50", "mean_separation"]
    pd.set_option("display.width", 220)

    for exp in aw.experiment.unique():
        sub = aw[aw.experiment == exp]
        print(f"\n=== {exp} ===")
        if sub.run_number.nunique() > 1:
            t = sub.groupby("channels")[cols].agg(["mean", "std"]).round(4)
        else:
            t = sub.set_index("channels")[cols].round(4)
        display(t)


=== v2_baseline_ens ===


,gap_iou,gap_f1,gap_ratio,detection_rate_c90_s90,merge_rate_c90_s90,detection_rate_c50_s50,mean_separation
channels,,,,,,,
3,0.4285,0.5999,0.9098,0.3816,0.2489,0.6259,0.7310
4,0.4706,0.6400,0.9697,0.2851,0.4163,0.4253,0.5387
7,0.5097,0.6752,1.0681,0.4781,0.2640,0.6757,0.7592



=== v8_coslr_ens ===


,gap_iou,gap_f1,gap_ratio,detection_rate_c90_s90,merge_rate_c90_s90,detection_rate_c50_s50,mean_separation
channels,,,,,,,
3,0.4146,0.5862,1.0239,0.3695,0.2836,0.6033,0.7040
4,0.4630,0.6329,0.9638,0.1719,0.5505,0.2609,0.3483
7,0.4832,0.6516,1.1360,0.3454,0.3514,0.5294,0.6320



=== v2_yaw_correction-epsV1 ===


gap_iou          gap_f1         gap_ratio         detection_rate_c90_s90         merge_rate_c90_s90         detection_rate_c50_s50         mean_separation        
            mean     std    mean     std      mean     std                   mean     std               mean     std                   mean     std            mean     std
channels                                                                                                                                                                   
3         0.4101  0.0101  0.5816  0.0102    0.9406  0.0859                 0.3376  0.0198             0.2718  0.0185                 0.5722  0.0150          0.6818  0.0259
4         0.4477  0.0087  0.6185  0.0083    1.0347  0.0501                 0.2600  0.0135             0.4124  0.0225                 0.3985  0.0159          0.5134  0.0154
7         0.4825  0.0135  0.6508  0.0122    1.1252  0.0896                 0.4471  0.0316             0.2600  0.0532                 0.6468  0.0480          0.7346  0.0480

## 4. Per-class stone separation (final ensemble) — the ashlar question

In [5]:
if gs is not None:
    ens = gs[(gs.experiment == "v2_baseline_ens") & (gs.wall == "AllWalls")]
    rows = []
    for _, r in ens.iterrows():
        for cls in ("Ashlar", "Polygonal", "Quarry"):
            if f"{cls}_n" in r and not pd.isna(r.get(f"{cls}_n")):
                rows.append({"channels": r.channels, "class": cls,
                             "n": int(r[f"{cls}_n"]),
                             "detection_rate": r[f"{cls}_detection_rate_c90_s90"],
                             "merge_rate": r[f"{cls}_merge_rate_c90_s90"],
                             "mean_coverage": r[f"{cls}_mean_coverage"],
                             "mean_separation": r[f"{cls}_mean_separation"]})
    t = pd.DataFrame(rows).pivot(index="class", columns="channels",
                                 values=["detection_rate", "merge_rate", "mean_separation"]).round(3)
    display(t)
    print("\nReading: high merge_rate + low mean_separation = stones fused together.")
    print("If 4ch ashlar shows this while 3ch does not, the visual impression is confirmed.")

detection_rate               merge_rate               mean_separation              
channels               3      4      7          3      4      7               3      4      7
class                                                                                        
Ashlar             0.359  0.382  0.503      0.356  0.523  0.362           0.651  0.549  0.708
Polygonal          0.591  0.237  0.626      0.136  0.424  0.182           0.862  0.522  0.843
Quarry             0.094  0.077  0.154      0.120  0.085  0.111           0.777  0.531  0.779


Reading: high merge_rate + low mean_separation = stones fused together.
If 4ch ashlar shows this while 3ch does not, the visual impression is confirmed.
